Training the Classifier and Testing on Asian Tales. First, train a classifier on Western folktales that have ATU labels. Then, apply that classifier to Asian folktales and see how confident it is. The classifier should be less confident on Asian tales, showing that ATU categories don't work well for non-Western stories.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import yaml
import logging
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import FolktaleDataLoader
from src.model import FolktaleClassifier
from src.visualization import (
    plot_confidence_distribution,
    plot_confidence_comparison
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Load configuration
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print('Setup complete.')

Phase 1: Load and Prepare Training Data

In [ ]:
# Load data
loader = FolktaleDataLoader('../config/config.yaml')

try:
    western_df = loader.load_western_tales()
    print(f"Loaded {len(western_df)} Western folktales")
    
    # Preprocess texts
    western_df['text_processed'] = western_df['text'].apply(loader.preprocess_text)
    
    # Prepare training/validation split
    from sklearn.model_selection import train_test_split
    
    # Assuming 'atu_category' or similar label column
    if 'atu_category' not in western_df.columns:
        # Fill with placeholder if column doesn't exist
        print("Warning: 'atu_category' column not found. Using placeholder.")
        western_df['atu_category'] = [f'type_{i % 10}' for i in range(len(western_df))]
    
    texts = western_df['text_processed'].values
    labels = western_df['atu_category'].values
    
    # Split data
    test_size = config['training']['test_size']
    val_size = config['training']['validation_size']
    
    texts_train, texts_test, labels_train, labels_test = train_test_split(
        texts, labels, test_size=test_size, random_state=config['model']['random_state']
    )
    
    texts_train, texts_val, labels_train, labels_val = train_test_split(
        texts_train, labels_train, test_size=val_size, random_state=config['model']['random_state']
    )
    
    print(f"Training set: {len(texts_train)} tales")
    print(f"Validation set: {len(texts_val)} tales")
    print(f"Test set: {len(texts_test)} tales")
    
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Cannot proceed without Western folktales dataset.")

Train Classifier (Phase 1)

In [ ]:
# Initialize classifier
clf = FolktaleClassifier(config, device='cpu')  # Use 'cuda' if GPU available

# Train classifier
print("Training logistic regression classifier...")
metrics = clf.train(
    texts=texts_train,
    labels=labels_train,
    validation_texts=texts_val,
    validation_labels=labels_val
)

for key, value in metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

# Save classifier
output_dir = Path('../results/models')
output_dir.mkdir(parents=True, exist_ok=True)
clf.save(str(output_dir / 'folktale_classifier.pkl'))
print(f"\n✓ Classifier saved to {output_dir / 'folktale_classifier.pkl'}")

Evaluate on Western Test Set

In [ ]:
# Test on Western folktales
predictions_west, probabilities_west = clf.predict(texts_test)
confidence_metrics_west = clf.get_confidence_metrics(probabilities_west)

print(f"Mean Confidence: {confidence_metrics_west['mean_confidence']:.4f}")
print(f"Std Confidence: {confidence_metrics_west['std_confidence']:.4f}")
print(f"Mean Entropy: {confidence_metrics_west['mean_entropy']:.4f}")
print(f"Std Entropy: {confidence_metrics_west['std_entropy']:.4f}")

# Plot confidence distribution
plot_confidence_distribution(
    confidence_metrics_west['max_confidence'],
    confidence_metrics_west['entropy'],
    label='Western Folktales (Test Set)',
    output_path='../results/plots/02_western_confidence.png'
)

Phase 2: Test on Asian/SE Asian Folktales

In [ ]:
# Load Asian folktales
try:
    asian_df = loader.load_asian_tales()
    
    if len(asian_df) > 0:
        # Preprocess texts
        asian_df['text_processed'] = asian_df['text'].apply(loader.preprocess_text)
        texts_asian = asian_df['text_processed'].values
        
        print(f"Loaded {len(texts_asian)} Asian/SE Asian folktales")
        
        # Test classifier on Asian folktales
        predictions_asian, probabilities_asian = clf.predict(texts_asian)
        confidence_metrics_asian = clf.get_confidence_metrics(probabilities_asian)
        
        print(f"Mean Confidence: {confidence_metrics_asian['mean_confidence']:.4f}")
        print(f"Std Confidence: {confidence_metrics_asian['std_confidence']:.4f}")
        print(f"Mean Entropy: {confidence_metrics_asian['mean_entropy']:.4f}")
        print(f"Std Entropy: {confidence_metrics_asian['std_entropy']:.4f}")
        
        # Plot confidence distribution
        plot_confidence_distribution(
            confidence_metrics_asian['max_confidence'],
            confidence_metrics_asian['entropy'],
            label='Asian/SE Asian Folktales',
            output_path='../results/plots/02_asian_confidence.png'
        )
    else:
        print("No Asian folktales loaded. Skipping Phase 2 testing.")
        confidence_metrics_asian = None
        
except Exception as e:
    print(f"Error loading Asian folktales: {e}")
    confidence_metrics_asian = None

Hypothesis Testing: Confidence Comparison

In [ ]:
if confidence_metrics_asian is not None:
    # Compare confidence
    west_conf = confidence_metrics_west['mean_confidence']
    asian_conf = confidence_metrics_asian['mean_confidence']
    
    print(f"Western Confidence (Mean): {west_conf:.4f}")
    print(f"Asian Confidence (Mean):    {asian_conf:.4f}")
    print(f"Difference:                 {west_conf - asian_conf:.4f}")
    
    # Statistical test
    from scipy import stats
    
    t_stat, p_value = stats.ttest_ind(
        confidence_metrics_west['max_confidence'],
        confidence_metrics_asian['max_confidence']
    )
    
    print(f"T-test results:")
    print(f"  t-statistic: {t_stat:.4f}")
    print(f"  p-value: {p_value:.2e}")
    
    if p_value < 0.05:
        print(f"Hypothesis SUPPORTED (p < 0.05):")
        print(f"  Classifier has significantly LOWER confidence on Asian folktales.")
        print(f"  This supports the hypothesis that ATU categories don't generalize well.")
    else:
        print(f"Hypothesis NOT supported (p >= 0.05):")
        print(f"  Classifier confidence is similar across Western and Asian folktales.")
    
    # Plot comparison
    plot_confidence_comparison(
        confidence_metrics_west['max_confidence'],
        confidence_metrics_asian['max_confidence'],
        output_path='../results/plots/02_confidence_comparison.png'
    )
    
    # Save results
    results = {
        'western_metrics': confidence_metrics_west,
        'asian_metrics': confidence_metrics_asian,
        'statistical_test': {
            't_statistic': float(t_stat),
            'p_value': float(p_value)
        }
    }
    
    import json
    with open('../results/analysis/02_confidence_analysis.json', 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"Results saved to ../results/analysis/02_confidence_analysis.json")
else:
    print("Cannot perform hypothesis test without Asian folktales data.")

Summary and Next Steps

In [ ]:
print(f"Trained logistic regression classifier on {len(texts_train)} Western folktales")
if confidence_metrics_asian is not None:
    print(f"Tested classifier on {len(texts_asian)} Asian/SE Asian folktales")
    print(f"Confidence difference: {west_conf - asian_conf:.4f} (Western > Asian)")
print("Next: Run notebook 03_interpretability.ipynb to:")
print("  1. Cluster Asian folktales using SAE-lens or k-means")
print("  2. Extract narrative motifs and archetypes")
print("  3. Analyze distinctive features per cluster")